# V6 — Strict Point-in-Time Replay with Selected V4.2

Replay the locked A2 holdout in global timestamp order. Compare the controlled static V4 ablation, adaptive V4, V4 with popularity, and selected V4.2 author affinity under identical catalog eligibility and seen-item masks. No model is trained in this notebook.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile
DRIVE_ROOT=Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT=DRIVE_ROOT/'data/gold/adaptive/v1'
BASE_GOLD_ROOT=DRIVE_ROOT/'data/gold/two_tower/v1_colab'
PROTOCOL_DIR=ADAPTIVE_ROOT/'evaluation_protocol/v2'
OUTPUT_DIR=DRIVE_ROOT/'artifacts/adaptive/v6_v4_2_replay'
WORK_ROOT=Path('/content/recsys_v6_v4_2')
SOURCE_ROOT=WORK_ROOT/'runtime'; V4_DIR=WORK_ROOT/'v4'; REPORT_DIR=WORK_ROOT/'report'
for path in (SOURCE_ROOT,V4_DIR,REPORT_DIR,OUTPUT_DIR): path.mkdir(parents=True,exist_ok=True)
print('report directory:',OUTPUT_DIR)

In [ ]:
required={'pyarrow':'pyarrow','pandas':'pandas','torch':'torch','huggingface_hub':'huggingface_hub>=0.32'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import pandas as pd, torch
assert torch.cuda.is_available(),'Select an L4 or A100 GPU runtime.'
required_paths=[ADAPTIVE_ROOT/'sequence_examples/train',ADAPTIVE_ROOT/'sequence_examples/validation',BASE_GOLD_ROOT/'item_features/static',PROTOCOL_DIR/'queries.parquet']
missing_paths=[str(path) for path in required_paths if not path.exists()]
assert not missing_paths,'Missing inputs:\n'+'\n'.join(missing_paths)
print('GPU:',torch.cuda.get_device_name(0))
print('input preflight: PASS')

In [ ]:
from huggingface_hub import hf_hub_download
HF_REPO_ID='khoa1505/recsys-artifacts'
RUNTIME_EXPECTED_SHA256='e404f8e03a6b7016f9598873bb8cd66b28c439dc5bc664a7b2fdadef93a1db19'
V4_EXPECTED_SHA256='4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'
def normalize_hf_token(raw):
    if isinstance(raw,bytes): raw=raw.decode('utf-8')
    if isinstance(raw,str):
        raw=raw.strip(); return raw if raw.startswith('hf_') else None
    if isinstance(raw,dict):
        preferred=('HF_TOKEN','token','value','secret')
        values=[raw[key] for key in preferred if key in raw]
        values.extend(value for key,value in raw.items() if key not in preferred)
        for value in values:
            token=normalize_hf_token(value)
            if token: return token
    if isinstance(raw,(list,tuple)):
        for value in raw:
            token=normalize_hf_token(value)
            if token: return token
    return None
try: HF_TOKEN=normalize_hf_token(userdata.get('HF_TOKEN'))
except Exception as exc:
    print('Colab Secret unavailable:',type(exc).__name__); HF_TOKEN=None
if not HF_TOKEN: HF_TOKEN=normalize_hf_token(getpass('Paste Hugging Face READ token: '))
assert HF_TOKEN,'HF_TOKEN must begin with hf_.'
runtime_zip=Path(hf_hub_download(repo_id=HF_REPO_ID,filename='runtime/adaptive_eval_runtime.zip',repo_type='model',token=HF_TOKEN))
v4_cached=Path(hf_hub_download(repo_id=HF_REPO_ID,filename='v4/v4_inference_fp16.pt',repo_type='model',token=HF_TOKEN))
def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        while block:=stream.read(8*1024*1024): digest.update(block)
    return digest.hexdigest()
assert sha256(runtime_zip)==RUNTIME_EXPECTED_SHA256,'Runtime ZIP hash mismatch'
assert sha256(v4_cached)==V4_EXPECTED_SHA256,'V4 checkpoint hash mismatch'
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None; archive.extractall(SOURCE_ROOT)
v4_target=V4_DIR/'best_model.pt'
if v4_target.exists() or v4_target.is_symlink(): v4_target.unlink()
v4_target.symlink_to(v4_cached.resolve())
REPLAY_SCRIPT=SOURCE_ROOT/'scripts/run_adaptive_v6_v4_2_replay.py'
assert REPLAY_SCRIPT.exists(); compile(REPLAY_SCRIPT.read_text(),str(REPLAY_SCRIPT),'exec')
print('artifact integrity: PASS')

## Run the locked V6 replay

Selected V4.2 parameters are fixed from notebook 25. This replay does not tune them again.

In [ ]:
if REPORT_DIR.exists(): shutil.rmtree(REPORT_DIR)
REPORT_DIR.mkdir(parents=True)
command=[sys.executable,str(REPLAY_SCRIPT),'--adaptive-root',str(ADAPTIVE_ROOT),'--base-gold-root',str(BASE_GOLD_ROOT),'--v4-dir',str(V4_DIR),'--protocol-dir',str(PROTOCOL_DIR),'--output-dir',str(REPORT_DIR),'--author-weight','0.2','--popularity-weight','0.02','--recency-tau','3.0','--calibration-user-fraction','0.35','--query-batch-size','16','--device','cuda']
replay_log=Path('/content/v6_v4_2_replay.log')
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with replay_log.open('w') as log:
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
code=process.wait()
if code!=0: raise RuntimeError('V6 replay failed:\n'+'\n'.join(replay_log.read_text(errors='replace').splitlines()[-200:]))
required_outputs=[REPORT_DIR/'v6_online_replay.json',REPORT_DIR/'online_replay_summary.csv',REPORT_DIR/'online_replay_segments.csv',REPORT_DIR/'recommendation_shift_examples.csv']
assert all(path.exists() for path in required_outputs),'Replay finished without all required reports.'

In [ ]:
summary=pd.read_csv(REPORT_DIR/'online_replay_summary.csv')
report=json.loads((REPORT_DIR/'v6_online_replay.json').read_text())
display(summary[['variant','hitrate@10','hitrate@100','hitrate@1000','ndcg@1000']])
print('point-in-time validation:',report['point_in_time_validation'])
print('V4.2 vs V4 HR@1000:',report['paired_user_bootstrap']['v4_2_vs_v4_adaptive']['hitrate@1000'])
print('V4.2 vs same-pop V4 HR@1000:',report['paired_user_bootstrap']['v4_2_vs_same_popularity_v4']['hitrate@1000'])
print('offline throughput:',report['latency']['exact_scoring_queries_per_second'],'queries/s')
print(report['v3_status'])

In [ ]:
archive_path=Path('/content/v6_v4_2_online_replay_report.zip')
if archive_path.exists(): archive_path.unlink()
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in REPORT_DIR.rglob('*'):
        if path.is_file(): archive.write(path,Path('v6_v4_2_replay')/path.relative_to(REPORT_DIR))
    archive.write(replay_log,Path('logs')/replay_log.name)
persistent=OUTPUT_DIR/archive_path.name
shutil.copy2(archive_path,persistent)
print('report:',persistent,archive_path.stat().st_size,'bytes')
files.download(str(archive_path))